In [1]:
import h5py
import numpy as np
import torch
import torch.nn as nn

import torch.functional as F

from sklearn.model_selection import train_test_split
from tqdm import tqdm
from torch.utils.data import DataLoader, Subset, Dataset
import torchmetrics

from sklearn.utils.class_weight import compute_class_weight
from collections import Counter

from sklearn.metrics import classification_report

In [2]:
import warnings
warnings.filterwarnings('ignore')

In [3]:
def train_epoch(model, optimizer, train_loader, loss_fn, n_classes):

    loss_log = []
    acc_log = []
    f1_log = []
    
    model.train()

    acc_calc = torchmetrics.classification.Accuracy(
        num_classes=n_classes, task="multiclass"
    ).to(device)

    f1_calc = torchmetrics.classification.F1Score(
        num_classes=n_classes, task="multiclass", average="macro"
    ).to(device)

    for spec, label in train_loader:
        spec = spec.to(device).float()
        label = label.to(device).long()

        optimizer.zero_grad()
        output_log = model(spec)
        output = torch.argmax(output_log, dim=1)

        loss = loss_fn(output_log, label)
        loss_log.append(loss.item())
        
        #acc = acc_calc(output, label)
        acc_calc.update(output, label)

        #f1 = f1_calc(output, label)
        f1_calc.update(output, label)

        loss.backward()
        optimizer.step()

    acc = acc_calc.compute().item()
    f1 = f1_calc.compute().item()

    acc_calc.reset()
    f1_calc.reset()
        
    return np.mean(loss_log), acc, f1

def test(model, loader, loss_fn, n_classes):
    
    loss_log = []
    preds_log = []
    labels_log = []
    
    model.eval()
    
    acc_calc = torchmetrics.classification.Accuracy(
        num_classes=n_classes, task="multiclass"
    ).to(device)

    f1_calc = torchmetrics.classification.F1Score(
        num_classes=n_classes, task="multiclass", average="macro"
    ).to(device)

    with torch.no_grad():
        for spec, label in loader:
            spec = spec.to(device).float()
            label = label.to(device).long()
            labels_log.extend(list(label.cpu().numpy()))
            output_log = model(spec)
            output = torch.argmax(output_log, dim=1)
            preds_log.extend(list(output.cpu().numpy()))
    
            loss = loss_fn(output_log, label)
            loss_log.append(loss.item())

            #acc = acc_calc(output, label)
            acc_calc.update(output, label)
    
            #f1 = f1_calc(output, label)
            f1_calc.update(output, label)

    acc = acc_calc.compute().item()
    f1 = f1_calc.compute().item()

    acc_calc.reset()
    f1_calc.reset()
        
    return np.mean(loss_log), acc, f1, preds_log, labels_log

def train(model, optimizer, n_epoch, train_loader, val_loader, loss_fn, scheduler=None, n_classes=6):
    best_f1 = 0
    
    train_loss_log, train_acc_log, train_f1_log, val_loss_log, val_acc_log, val_f1_log = [], [], [], [], [], []

    for epoch in tqdm(range(n_epoch)):
        train_loss, train_acc, train_f1 = train_epoch(model, optimizer, train_loader, loss_fn, n_classes)
        val_loss, val_acc, val_f1, preds_log, labels_log = test(model, val_loader, loss_fn, n_classes)
        
        train_loss_log.append(train_loss)
        train_acc_log.append(train_acc)
        train_f1_log.append(train_f1)

        val_loss_log.append(val_loss)
        val_acc_log.append(val_acc)
        val_f1_log.append(val_f1)

        print(f"Epoch {epoch}")
        print(f" train loss: {train_loss}, train acc: {train_acc}, train f1: {train_f1}")
        print(f" val loss: {val_loss}, val acc: {val_acc}, val f1: {val_f1}\n")
        print(classification_report(np.array(labels_log), np.array(preds_log)))

        if scheduler is not None:
            if isinstance(scheduler,torch.optim.lr_scheduler.ReduceLROnPlateau):
                scheduler.step(val_f1)
            else:
                scheduler.step()


        if val_f1 > best_f1:
            best_f1 = val_f1
            torch.save(model.state_dict(), "best_model.pt")

    return train_loss_log, train_acc_log, train_f1_log, val_loss_log, val_acc_log, val_f1_log

# Подготовка датасета

In [6]:
path = '/kaggle/input/datasets/alexandra841/features-audio-embeddings/features_audio_embeddings_upd.h5'

In [7]:
with h5py.File(path, 'r') as f:
    x_rnn = f['x_rnn'][:]
    x_mlp = f['x_mlp'][:]
    label = f['class_mood_int'][:]

In [8]:
class Dataset(Dataset):

    def __init__(self, seq, label):
        self.seq = seq
        self.label = label

    def __len__(self):
        return len(self.label)

    def __getitem__(self, idx):

        seq = (self.seq)[idx]
        label = (self.label)[idx]
        
        seq = torch.tensor(seq)
        label = torch.tensor(label)

        return seq, label

In [9]:
dataset = Dataset(x_rnn, label)
labels = dataset[:][1]

In [10]:
torch.unique(labels, return_counts=True)

(tensor([0, 1, 2, 3, 4, 5], dtype=torch.int8),
 tensor([4326, 1987, 5126, 2452,  793,  919]))

In [11]:
train_val_idx, test_idx = train_test_split(
    np.arange(len(dataset)), test_size=0.2, shuffle=True, random_state=42, stratify=labels
)

trainvalset = torch.utils.data.Subset(dataset, train_val_idx)
testset = torch.utils.data.Subset(dataset, test_idx)

labels_train = labels[train_val_idx]

In [12]:
train_idx, val_idx = train_test_split(
    np.arange(len(trainvalset)), test_size=0.2, shuffle=True, random_state=42, stratify=labels_train
)

trainset = torch.utils.data.Subset(trainvalset, train_idx)
valset = torch.utils.data.Subset(trainvalset, val_idx)

In [13]:
train_loader = DataLoader(trainset, batch_size=32, shuffle=True, num_workers=4, drop_last=True)
val_loader = DataLoader(valset, batch_size=32, shuffle=False, num_workers=4)
test_loader = DataLoader(testset, batch_size=32, shuffle=False,  num_workers=4)

In [14]:
n_classes = 6
in_channels = 1

In [15]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


In [16]:
class_weights = compute_class_weight(
        'balanced',
        classes=np.arange(n_classes),
        y=labels_train.numpy()
    )

In [17]:
class_weights_tensor = torch.tensor(
    class_weights, dtype=torch.float32
).to(device)

class_weights_tensor

tensor([0.6011, 1.3084, 0.5073, 1.0609, 3.2813, 2.8304], device='cuda:0')

In [18]:
weights = torch.tensor(class_weights, dtype=torch.float)
weights = torch.sqrt(weights)

class_weights_tensor_soft = torch.tensor(
    weights, dtype=torch.float32
).to(device)

# Модель

In [24]:
class MusicBert(nn.Module): 
    def __init__(self, n_classes=6): 
        super().__init__() 

        self.norm = nn.LayerNorm(768) 
        self.cls_token = nn.Parameter(torch.randn(1, 1, 768)) # так как еще + cls_token 
        self.position_embedding = nn.Embedding(257, 768) 
        self.embedding_dropout = nn.Dropout(0.1)
        
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=768,
            nhead=8,
            dim_feedforward=2048,
            dropout=0.2,
            activation='gelu',
            batch_first=True,
            norm_first=True
        )
        
        self.encoder = nn.TransformerEncoder( 
            encoder_layer, 
            num_layers = 2
        ) 

        self.final_norm = nn.LayerNorm(768)

        self.attention_pool = nn.Sequential(
            nn.Linear(768, 128),
            nn.Tanh(),
            nn.Linear(128, 1)
        )
        
        self.fc = nn.Sequential(
            nn.Linear(768, 512),
            nn.GELU(),
            nn.Dropout(0.3),

            nn.Linear(512, 256),
            nn.GELU(),
            nn.Dropout(0.2),

            nn.Linear(256, n_classes)
        )
        
    def forward(self, x): 
        batch_size = x.size(0)
        
        x = self.norm(x) 
        cls = self.cls_token.expand(batch_size, -1, -1) 
        x = torch.cat([cls, x], dim=1) 
        
        positions = torch.arange(0, x.size(1), device=x.device) 
        pos = self.position_embedding(positions) 
        
        x = x + pos
        x = self.embedding_dropout(x)
        x = self.encoder(x)
        x = self.final_norm(x)
        attn_weights = self.attention_pool(x)

        attn_weights = torch.softmax(
            attn_weights,
            dim=1
        )

        pooled = (
            attn_weights * x
        ).sum(dim=1)
        
        return self.fc(pooled)

In [25]:
loss_fn = nn.CrossEntropyLoss(weight=class_weights_tensor_soft)
#label_smoothing=0.1

In [29]:
bert_model = MusicBert()
bert_model = bert_model.to(device)

In [30]:
optimizer = torch.optim.AdamW(
    bert_model.parameters(),
    lr=1e-4,
    weight_decay=1e-4
)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=20
)

In [31]:
train_loss_log, train_acc_log, train_f1_log, val_loss_log, val_acc_log, val_f1_log = train(
    bert_model,
    optimizer,
    20,
    train_loader, 
    val_loader, 
    loss_fn,
    scheduler
)

  5%|▌         | 1/20 [01:17<24:36, 77.72s/it]

Epoch 0
 train loss: 1.6325832826968951, train acc: 0.40524840354919434, train f1: 0.2318122386932373
 val loss: 1.5771381643754017, val acc: 0.43412095308303833, val f1: 0.2621709108352661

              precision    recall  f1-score   support

           0       0.40      0.76      0.52       692
           1       0.23      0.02      0.03       318
           2       0.56      0.49      0.52       821
           3       0.38      0.34      0.36       392
           4       0.15      0.03      0.05       127
           5       0.18      0.05      0.08       147

    accuracy                           0.43      2497
   macro avg       0.32      0.28      0.26      2497
weighted avg       0.40      0.43      0.38      2497



 10%|█         | 2/20 [02:34<23:07, 77.06s/it]

Epoch 1
 train loss: 1.5276722472447615, train acc: 0.43739983439445496, train f1: 0.3205517530441284
 val loss: 1.551230420794668, val acc: 0.44453343749046326, val f1: 0.309476375579834

              precision    recall  f1-score   support

           0       0.42      0.70      0.53       692
           1       0.31      0.15      0.20       318
           2       0.59      0.49      0.53       821
           3       0.38      0.38      0.38       392
           4       0.25      0.02      0.04       127
           5       0.21      0.14      0.17       147

    accuracy                           0.44      2497
   macro avg       0.36      0.32      0.31      2497
weighted avg       0.43      0.44      0.42      2497



 15%|█▌        | 3/20 [03:50<21:44, 76.76s/it]

...

Epoch 17
 train loss: 0.20076951887219763, train acc: 0.9162660241127014, train f1: 0.9348374605178833
 val loss: 3.933710549421287, val acc: 0.40528634190559387, val f1: 0.30723828077316284

              precision    recall  f1-score   support

           0       0.44      0.53      0.48       692
           1       0.23      0.24      0.23       318
           2       0.53      0.49      0.51       821
           3       0.38      0.35      0.36       392
           4       0.14      0.11      0.12       127
           5       0.14      0.12      0.13       147

    accuracy                           0.41      2497
   macro avg       0.31      0.31      0.31      2497
weighted avg       0.40      0.41      0.40      2497



 95%|█████████▌| 19/20 [24:13<01:16, 76.48s/it]

Epoch 18
 train loss: 0.18620176241995814, train acc: 0.9233773946762085, train f1: 0.9403737783432007
 val loss: 4.053450751894955, val acc: 0.4116940200328827, val f1: 0.3060353994369507

              precision    recall  f1-score   support

           0       0.45      0.53      0.48       692
           1       0.24      0.23      0.23       318
           2       0.52      0.52      0.52       821
           3       0.37      0.35      0.36       392
           4       0.15      0.09      0.12       127
           5       0.15      0.11      0.12       147

    accuracy                           0.41      2497
   macro avg       0.31      0.30      0.31      2497
weighted avg       0.40      0.41      0.40      2497



100%|██████████| 20/20 [25:30<00:00, 76.52s/it]

Epoch 19
 train loss: 0.1825009522219308, train acc: 0.9240785241127014, train f1: 0.9417916536331177
 val loss: 4.039129412614945, val acc: 0.4084901809692383, val f1: 0.30309516191482544

              precision    recall  f1-score   support

           0       0.45      0.50      0.47       692
           1       0.23      0.24      0.23       318
           2       0.52      0.53      0.53       821
           3       0.36      0.35      0.36       392
           4       0.15      0.09      0.11       127
           5       0.13      0.10      0.11       147

    accuracy                           0.41      2497
   macro avg       0.31      0.30      0.30      2497
weighted avg       0.40      0.41      0.40      2497

